# Análisis Exploratorio de Datos — Sistema de Predicción de Dengue RD

**República Dominicana | MSP-DIGEPI | Ensemble RF + LSTM**

Este notebook realiza el análisis exploratorio completo de los datos:
- Distribución geográfica de casos por provincia
- Estacionalidad y patrones temporales
- Correlaciones entre variables climáticas y epidemiológicas
- Análisis del índice ENSO y su impacto
- Feature importance del modelo Random Forest

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

from src.data.ingestion import DataIngestion
from config.settings import PROVINCES

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✓ Librerías cargadas correctamente')

## 1. Carga de Datos

In [ ]:
ingestion = DataIngestion()
df = ingestion.consolidate_all_sources(weeks=208)  # 4 años de datos
print(f'Dataset maestro: {df.shape[0]:,} registros × {df.shape[1]} columnas')
df.head()

In [ ]:
print('Resumen estadístico:')
df.describe().round(2)

In [ ]:
# Valores nulos
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
if len(null_pct) > 0:
    print('Variables con valores nulos:')
    print(null_pct.to_string())
else:
    print('✓ Sin valores nulos en el dataset')

## 2. Distribución del Índice de Riesgo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['outbreak_risk_index'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(65, color='red', linestyle='--', label='Umbral Epidemia (65)')
axes[0].axvline(df['outbreak_risk_index'].mean(), color='orange', linestyle='--', 
                label=f"Media ({df['outbreak_risk_index'].mean():.1f})")
axes[0].set_xlabel('Índice de Riesgo de Brote')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Índice de Riesgo')
axes[0].legend()

# Box plot por categoría de riesgo
def classify(r):
    if r < 25: return 'Bajo'
    if r < 50: return 'Moderado'
    if r < 65: return 'Alto'
    if r < 80: return 'Epidemia'
    return 'Crítico'

df['risk_level'] = df['outbreak_risk_index'].apply(classify)
level_counts = df['risk_level'].value_counts()
colors = {'Bajo': '#28a745', 'Moderado': '#ffc107', 'Alto': '#fd7e14', 
           'Epidemia': '#dc3545', 'Crítico': '#6f0000'}
bars = axes[1].bar(level_counts.index, level_counts.values, 
                   color=[colors.get(l, 'gray') for l in level_counts.index])
axes[1].set_xlabel('Nivel de Riesgo')
axes[1].set_ylabel('Número de registros')
axes[1].set_title('Distribución por Nivel de Riesgo')
for bar, val in zip(bars, level_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
                f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../visuals/01_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: visuals/01_risk_distribution.png')

## 3. Riesgo Promedio por Provincia

In [ ]:
prov_risk = df.groupby('province')['outbreak_risk_index'].agg(['mean', 'max', 'std']).round(2)
prov_risk = prov_risk.sort_values('mean', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(prov_risk.index, prov_risk['mean'],
               xerr=prov_risk['std'], error_kw={'capsize': 3},
               color=['#dc3545' if v >= 65 else '#fd7e14' if v >= 50 else 
                      '#ffc107' if v >= 25 else '#28a745' for v in prov_risk['mean']])
ax.axvline(65, color='red', linestyle='--', alpha=0.5, label='Umbral Epidemia (65)')
ax.set_xlabel('Índice de Riesgo Promedio')
ax.set_title('Riesgo Promedio de Dengue por Provincia (2020-2024)\nRepública Dominicana')
ax.legend()
plt.tight_layout()
plt.savefig('../visuals/02_risk_by_province.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Estacionalidad — Patrón Temporal

In [ ]:
if 'week' in df.columns:
    weekly_avg = df.groupby('week')['outbreak_risk_index'].mean()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(weekly_avg.index, weekly_avg.values, alpha=0.3, color='steelblue')
    ax.plot(weekly_avg.index, weekly_avg.values, color='steelblue', linewidth=2.5, label='Riesgo promedio')
    ax.axhline(65, color='red', linestyle='--', alpha=0.7, label='Umbral Epidemia (65)')
    ax.axvspan(18, 44, alpha=0.08, color='red', label='Temporada lluviosa (may-oct)')
    ax.set_xlabel('Semana Epidemiológica')
    ax.set_ylabel('Índice de Riesgo Promedio')
    ax.set_title('Estacionalidad del Riesgo de Dengue — República Dominicana')
    ax.legend()
    ax.set_xlim(1, 52)
    plt.tight_layout()
    plt.savefig('../visuals/03_seasonality.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Pico de riesgo en semana:', weekly_avg.idxmax(), f'({weekly_avg.max():.1f})')

## 5. Correlaciones entre Variables

In [ ]:
corr_cols = ['outbreak_risk_index', 'rainfall_mm', 'temp_avg_c', 'humidity_pct',
             'cases', 'incidence_rate_100k', 'population_density_km2', 'poverty_index']
available = [c for c in corr_cols if c in df.columns]
corr_matrix = df[available].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlación — Variables del Modelo de Dengue RD')
plt.tight_layout()
plt.savefig('../visuals/04_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

target_corr = corr_matrix['outbreak_risk_index'].drop('outbreak_risk_index').sort_values(key=abs, ascending=False)
print('\nCorrelación con el índice de riesgo:')
print(target_corr.to_string())

## 6. Análisis ENSO vs Riesgo de Dengue

In [ ]:
if 'enso_index' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot
    sc = axes[0].scatter(df['enso_index'], df['outbreak_risk_index'],
                         c=df['outbreak_risk_index'], cmap='RdYlGn_r',
                         alpha=0.3, s=10)
    plt.colorbar(sc, ax=axes[0], label='Índice de Riesgo')
    z = np.polyfit(df['enso_index'].fillna(0), df['outbreak_risk_index'].fillna(0), 1)
    p = np.poly1d(z)
    x_range = np.linspace(df['enso_index'].min(), df['enso_index'].max(), 100)
    axes[0].plot(x_range, p(x_range), 'r--', linewidth=2, label='Tendencia')
    axes[0].axvline(0, color='gray', linestyle=':', alpha=0.5)
    axes[0].set_xlabel('Índice ENSO (ONI)')
    axes[0].set_ylabel('Índice de Riesgo')
    axes[0].set_title('ENSO vs Riesgo de Dengue')
    axes[0].legend()
    
    # Boxplot por fase ENSO
    df['enso_phase'] = pd.cut(df['enso_index'], 
                               bins=[-3, -0.5, 0.5, 3],
                               labels=['La Niña\n(< -0.5)', 'Neutral\n(-0.5 a 0.5)', 'El Niño\n(> 0.5)'])
    enso_risk = [df[df['enso_phase'] == p]['outbreak_risk_index'].dropna()
                 for p in df['enso_phase'].cat.categories]
    bp = axes[1].boxplot(enso_risk, labels=['La Niña', 'Neutral', 'El Niño'],
                          patch_artist=True)
    colors_enso = ['#5b9bd5', '#70ad47', '#ed7d31']
    for patch, color in zip(bp['boxes'], colors_enso):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1].set_ylabel('Índice de Riesgo')
    axes[1].set_title('Riesgo por Fase ENSO')
    
    plt.tight_layout()
    plt.savefig('../visuals/05_enso_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Conclusiones del EDA

In [ ]:
epidemic_pct = (df['outbreak_risk_index'] >= 65).mean() * 100
peak_week = df.groupby('week')['outbreak_risk_index'].mean().idxmax() if 'week' in df.columns else 'N/A'
highest_prov = df.groupby('province')['outbreak_risk_index'].mean().idxmax()

print('=' * 55)
print('CONCLUSIONES DEL ANÁLISIS EXPLORATORIO')
print('=' * 55)
print(f'Registros analizados: {len(df):,}')
print(f'Provincias cubiertas: {df["province"].nunique()}')
print(f'% del tiempo en nivel epidémico: {epidemic_pct:.1f}%')
print(f'Semana de mayor riesgo histórico: SE {peak_week}')
print(f'Provincia de mayor riesgo promedio: {highest_prov}')
print()
print('Variables más correlacionadas con el riesgo:')
if 'incidence_rate_100k' in df.columns:
    key_vars = df[['outbreak_risk_index', 'rainfall_mm', 'temp_avg_c', 
                   'humidity_pct', 'incidence_rate_100k']].corr()['outbreak_risk_index']
    key_vars = key_vars.drop('outbreak_risk_index').sort_values(key=abs, ascending=False)
    for var, corr in key_vars.items():
        print(f'  {var}: {corr:.3f}')
print('=' * 55)
print('✓ Análisis completado. Gráficos guardados en visuals/')